# SATURN: a strict species-stratified cell holdout

Ten percent of cells in each of the five species are excluded from preprocessing, pretraining, teacher graphs, fine-tuning and selection. The original architecture is restored: **200 macrogenes / 256 hidden / 256 embedding dimensions**. All four methods share 20 epochs of unlabelled training, then run 30 fine-tuning epochs each.

Run on the fresh GCP VM using the experiment environment and the canonical runbook. The entire VM window, including setup, is capped at six hours. Existing complete reports are detected automatically and can be viewed without training.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
from IPython.display import display, Image, HTML
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/run_cell_holdout_jax.py').is_file())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from jax_saturn.experiments.cell_holdout import HoldoutConfig
config = HoldoutConfig.from_env()
root = config.root
default_report_mode = '1' if (root / 'holdout_report.json').is_file() else '0'
report_only = os.environ.get('SATURN_HOLDOUT_REPORT_ONLY', default_report_mode) == '1'
display(pd.Series(config.scientific_record(), name='experiment'))


## Split and exposure audit

The split uses species and stable observation IDs only, with seed 0 and rounded-up test counts. Cell labels do not determine membership. The full externally pretrained protein representations are reused; all experiment-specific fitting uses training observations only. This experiment measures held-out cells within known species, with no donor-generalization claim.


In [ ]:
if not report_only:
    subprocess.run([sys.executable, repo_root / 'scripts/run_cell_holdout_jax.py', 'prepare'], check=True, cwd=repo_root)
split = pd.read_csv(root / 'split/split.csv', dtype=str, keep_default_na=False)
assert not split.obs_id.duplicated().any()
assert set(split['split']) == {'train', 'test'}
display(split.groupby(['species','split']).size().unstack())
display(pd.read_csv(root / 'split/split_composition.csv'))
display(json.loads((root / 'split/split_record.json').read_text())['counts'])


## Shared foundation and four fine-tuning methods

The controller fits training-only preprocessing and centroids, performs unlabelled pretraining, and trains the labelled baseline plus InfoNCE, MMD and partial OT. The three label-free methods receive only embeddings, macrogenes, species and observation IDs. Their checkpoint selection uses training-only species mixing and frozen-neighbour recall.

Two real production epochs establish stage timing; a conservative gate checks the remaining work against the original VM deadline. Checkpoints, logs and the partially executed notebook are preserved on interruption.


In [ ]:
if not report_only:
    start = os.environ.get('SATURN_VM_START')
    if not start:
        raise RuntimeError('Source SATURN_VM_START from the initial GCE lastStartTimestamp before cloud training.')
    subprocess.run([sys.executable, repo_root / 'scripts/run_cell_holdout_jax.py', 'run', '--vm-start-time', start, '--skip-notebook'], check=True, cwd=repo_root)
for trial in ('baseline','infonce','mmd','ot'):
    history = pd.read_csv(root / 'run' / trial / 'history.csv')
    assert history.epoch.tolist() == list(range(1, config.metric_epochs + 1))
    print(trial, 'completed epochs:', len(history))
display(pd.read_csv(root / 'run/comparison.csv'))
display(json.loads((root / 'run/acceptance.json').read_text()))


## Do held-out cells generalize well?

Compare training queries (self excluded) with held-out queries against the same training reference. Cell-type prediction uses a majority vote among 15 nearest training neighbours. The first plot shows vote accuracy, same-type neighbour agreement and species mixing. The next plots expose species and cell types whose averages hide a larger gap. Assess absolute accuracy and species mixing alongside the gap. These are descriptive results for a cell holdout within these input atlases, without an unseen-donor claim.

For a completed run, regenerate with `DEVICE=cpu NEIGHBOR_BACKEND=jax python scripts/run_cell_holdout_jax.py diagnostics`. This uses finalized embeddings and existing UMAP coordinates.

In [ ]:
diagnostic_root = root / 'diagnostics'
if (diagnostic_root / 'summary.json').is_file():
    for name in ('generalization_summary', 'generalization_by_species', 'generalization_by_cell_type'):
        path = diagnostic_root / f'{name}.png'
        if path.is_file():
            display(Image(filename=str(path)))
    generalization = pd.read_csv(diagnostic_root / 'generalization.csv')
    overall = generalization[generalization.dimension == 'overall'].set_index('trial')
    display(pd.DataFrame({
        'Training accuracy (%)': 100 * overall.train_label_vote_accuracy,
        'Held-out accuracy (%)': 100 * overall.test_label_vote_accuracy,
        'Test minus train (pp)': overall.gap_pp_label_vote_accuracy,
        'Held-out cross-species neighbours (%)': 100 * overall.test_species_mixing_fraction,
    }).round(2))
else:
    print('Generalization diagnostics have not been generated for this archived run yet.')

## Frozen inference and held-out placement

Test counts are aligned to saved training-selected genes, passed through the frozen pretrained macrogene encoder, then through each finalized metric encoder. Labels are joined only after inference. UMAP is fitted on training embeddings only and test cells are placed using `transform`. Training cells are grey; test cells are coloured by species or fine cell type. Coordinates across different methods are separate UMAP coordinate systems.


In [ ]:
display(Image(filename=str(root / 'report_assets/heldout_umaps.png')))


In [ ]:
import plotly.io as pio
# The standalone HTML preserves all points, legends and observation-level hover.
html_path = root / 'diagnostics/heldout_errors.html'
if not html_path.is_file():
    html_path = root / 'report_assets/heldout_umaps.html'
display(HTML(html_path.read_text()))


## Held-out metrics

Every test cell queries the frozen **training reference population**, using 15 cosine neighbours. Overall and per-species results show cross-species mixing and fine-label agreement. These metrics are post-hoc diagnostics and do not affect model or epoch selection. Scientific acceptance in the usual training comparison is recorded separately from execution completion.


In [ ]:
heldout = pd.read_csv(root / 'heldout_metrics.csv')
display(heldout[heldout.species == 'all'])
display(heldout[heldout.species != 'all'])
display(json.loads((root / 'holdout_report.json').read_text()))


## Reusable artifacts

The run contains frozen preprocessing, split identities and input hashes, all Orbax epochs, aligned test embedding NPZs, train/test AnnData, fitted UMAP reducers, coordinate NPZs, PNG/PDF figures and interactive HTML. A report can be regenerated with `python scripts/run_cell_holdout_jax.py report` in the same experiment environment. Verify the final bucket backup and local download, and confirm the VM is stopped.
